<a href="https://colab.research.google.com/github/Sagnik-Chowdhury/Federated-Learning-1/blob/Sourit/Fed_Trimmed_Mean_Aggregation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Robust Aggregation & Statistical Noise Modalities
## Comparing Trimmed Mean, Laplace Noise, and Gaussian Noise across Data Types

**Objective:**

This experiment evaluates how different data modalities respond to advanced Federated Learning defenses. We are testing two distinct dataset architectures:

1. **Integrated/Dense Data (MNIST):** High-dimensional, structured pixel data reliant on spatial relationships.

2. **Scattered/Tabular Data (Breast Cancer Dataset):** Low-dimensional, heterogeneous, and independent features.

**Defense Mechanisms Tested:**

* **Trimmed Mean:** A Byzantine-robust aggregation strategy that discards the top and bottom 5% of weight updates to filter out extreme outliers or poisoned data.

* **Statistical Noise Injection:** Adding mathematical noise to the aggregated server weights to preserve client anonymity. We compare **Laplace Noise** (heavy-tailed distributions, theoretically better suited for scattered tabular data) against standard **Gaussian/Normal Noise**.


## Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, TensorDataset
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
import copy
import pandas as pd
import matplotlib.pyplot as plt

## Architectures

In [ ]:
#  For Integrated Data (MNIST Images)
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        return self.fc4(x)


In [ ]:
class TabularNet(nn.Module):
    def __init__(self):
        super(TabularNet, self).__init__()
        # Reduced capacity: 30 -> 16 -> 8 -> 2
        self.fc1 = nn.Linear(30, 16)
        # Dropout randomly turns off 20% of neurons to prevent memorization
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x) # Apply dropout after the first layer
        x = self.relu(self.fc2(x))
        return self.fc3(x)

## Data Preparation

We simulate 20 clients as we trim 5% of top and bottom ,that is atleast 1 client(.05*20=1) .

In [ ]:
NUM_CLIENTS = 20

In [ ]:
print("Preparing MNIST Dataset (Integrated Data)")

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

mnist_full = datasets.MNIST('./data', train=True, download=True, transform=transform)
mnist_split = random_split(mnist_full, [len(mnist_full) // NUM_CLIENTS] * NUM_CLIENTS)
mnist_loaders = [DataLoader(ds, batch_size=32, shuffle=True) for ds in mnist_split]

mnist_test = datasets.MNIST('./data', train=False, download=True, transform=transform)
mnist_test_loader = DataLoader(mnist_test, batch_size=1000, shuffle=False)

print("\nData preparation complete")

In [ ]:
from sklearn.model_selection import train_test_split

print("Preparing Breast Cancer Dataset (Strict Train/Test Split)")

# 1. Load and scale the data
data = load_breast_cancer()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data.data)
y = data.target

# 2. THE FIX: Strictly separate 20% of the data for testing BEFORE tensor conversion
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 3. Convert to PyTorch Tensors
tabular_train = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
tabular_test = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

# 4. Split ONLY the training data among clients
tab_split_size = len(tabular_train) // NUM_CLIENTS
tab_splits = [tab_split_size] * NUM_CLIENTS
tab_splits[-1] += len(tabular_train) % NUM_CLIENTS
tabular_loaders = [DataLoader(ds, batch_size=8, shuffle=True) for ds in random_split(tabular_train, tab_splits)]

# 5. The Test Loader now holds data the clients have NEVER seen
tabular_test_loader = DataLoader(tabular_test, batch_size=len(tabular_test), shuffle=False)

print("\nData preparation complete")

## Defense Implementations: Trimmed Mean & Differential Privacy

To test the mentor's hypothesis, this section defines two distinct server-side operations:

1. **`trimmed_mean_aggregation`**: Sorts client updates, removes the top/bottom 5% extreme values per parameter, and averages the remaining 90%.

2. **`add_dp_noise`**: After aggregation, the server intentionally degrades the global model by injecting statistical noise. We can toggle between `normal` (Gaussian) and `laplace` to observe how the fatter tails of the Laplace distribution interact with the different neural network weights.

In [ ]:
def trimmed_mean_aggregation(client_weights_list, trim_ratio=0.05):
    """
    Sorts weights, drops the top and bottom trim_ratio (e.g., 5%),
    and averages the remaining values to prevent poisoning.
    """
    num_clients = len(client_weights_list)
    trim_count = int(trim_ratio * num_clients)

    # We must grab the FIRST client's dictionary to use as a template.
    global_weights = copy.deepcopy(client_weights_list[0])

    for key in global_weights.keys():
        # Stack all client tensors for this specific layer
        stacked_weights = torch.stack([client[key] for client in client_weights_list])

        if trim_count > 0:
            sorted_weights, _ = torch.sort(stacked_weights, dim=0)
            trimmed_weights = sorted_weights[trim_count : num_clients - trim_count]
        else:
            trimmed_weights = stacked_weights

        # Average the remaining safe weights
        global_weights[key] = torch.mean(trimmed_weights, dim=0)

    return global_weights

In [ ]:
def add_dp_noise(weights, noise_type='none', scale=0.01):
    """
    Injects Differential Privacy noise into the aggregated weights.
    Compares Laplace (good for sparse/scattered) vs Normal (Gaussian).
    """
    if noise_type == 'none':
        return weights

    noisy_weights = copy.deepcopy(weights)

    for key in noisy_weights.keys():
        tensor = noisy_weights[key]

        if noise_type == 'normal':
            # Gaussian Noise: N(0, scale^2)
            noise = torch.randn_like(tensor) * scale

        elif noise_type == 'laplace':
            # Laplace Noise: Lap(0, scale)
            m = torch.distributions.laplace.Laplace(torch.tensor([0.0]), torch.tensor([scale]))
            noise = m.sample(tensor.shape).squeeze(-1).to(tensor.device)

        noisy_weights[key] = tensor + noise

    return noisy_weights

### Federated Training & Evaluation

The loop below acts as our control center. By altering the `NOISE_TYPE` variable, we can simulate different privacy conditions and observe how the Tabular model degrades compared to the MNIST model.

In [ ]:
datasets_to_test = ['mnist', 'tabular']
noise_types_to_test = ['none', 'normal', 'laplace']
NOISE_SCALE = 0.05
federated_rounds = 5
epochs_per_round = 1

# Store final results for the table
experiment_results = {'mnist': {}, 'tabular': {}}

# Store per-round tracking for the plots
history = {
    'mnist': {n: {'test_acc': [], 'test_loss': [], 'train_loss': []} for n in noise_types_to_test},
    'tabular': {n: {'test_acc': [], 'test_loss': [], 'train_loss': []} for n in noise_types_to_test}
}

criterion_eval = nn.CrossEntropyLoss()

for dataset in datasets_to_test:
    print(f"\nTESTING DATASET: {dataset.upper()}")

    for noise in noise_types_to_test:
        print(f"\n[{noise.upper()} NOISE]")

        # 1. SETUP & RESET THE MODEL
        if dataset == 'mnist':
            global_model = MNISTNet()
            loaders = mnist_loaders
            test_loader = mnist_test_loader
            lr = 0.001
        else:
            global_model = TabularNet()
            loaders = tabular_loaders
            test_loader = tabular_test_loader
            lr = 0.01

        # 2. THE FEDERATED ROUNDS
        for round_num in range(federated_rounds):
            client_weights = []

            for client_idx in range(NUM_CLIENTS):
                local_model = MNISTNet() if dataset == 'mnist' else TabularNet()
                local_model.load_state_dict(global_model.state_dict())

                optimizer = optim.Adam(local_model.parameters(), lr=lr)
                criterion = nn.CrossEntropyLoss()

                local_model.train()
                for epoch in range(epochs_per_round):
                    for inputs, labels in loaders[client_idx]:
                        optimizer.zero_grad()
                        outputs = local_model(inputs)
                        loss = criterion(outputs, labels)
                        loss.backward()
                        optimizer.step()

                client_weights.append(local_model.state_dict())

            # Server Aggregation & Statistical Noise
            aggregated_weights = trimmed_mean_aggregation(client_weights, trim_ratio=0.05)
            secured_weights = add_dp_noise(aggregated_weights, noise_type=noise, scale=NOISE_SCALE)
            global_model.load_state_dict(secured_weights)

            # 3. ROUND EVALUATION (Train and Test Loss Tracking)
            global_model.eval()

            # --- Evaluate on Training Data ---
            train_loss_accum = 0.0
            train_total = 0
            with torch.no_grad():
                for client_loader in loaders:
                    for inputs, labels in client_loader:
                        outputs = global_model(inputs)
                        train_loss_accum += criterion_eval(outputs, labels).item() * inputs.size(0)
                        train_total += labels.size(0)
            avg_train_loss = train_loss_accum / train_total

            # --- Evaluate on Testing Data ---
            correct = 0
            test_total = 0
            test_loss_accum = 0.0
            with torch.no_grad():
                for inputs, labels in test_loader:
                    outputs = global_model(inputs)
                    test_loss_accum += criterion_eval(outputs, labels).item() * inputs.size(0)
                    _, predicted = torch.max(outputs.data, 1)
                    test_total += labels.size(0)
                    correct += (predicted == labels).sum().item()

            acc = 100 * correct / test_total
            avg_test_loss = test_loss_accum / test_total

            # Store metrics
            history[dataset][noise]['test_acc'].append(acc)
            history[dataset][noise]['test_loss'].append(avg_test_loss)
            history[dataset][noise]['train_loss'].append(avg_train_loss)

            # Print Round Metrics
            #print(f"   Round {round_num+1} | Acc: {acc:.2f}% | Test Loss: {avg_test_loss:.4f} | Train Loss: {avg_train_loss:.4f}")

        # 4. FINAL OUTPUT PER CONFIGURATION
        final_accuracy = history[dataset][noise]['test_acc'][-1]
        print(f"Final Accuracy ({noise.upper()}): {final_accuracy:.3f}%\n")
        experiment_results[dataset][noise] = round(final_accuracy, 3)

In [ ]:
print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)

results_df = pd.DataFrame(experiment_results).T
results_df.columns = ['Baseline (No Noise)', 'Normal (Gaussian)', 'Laplace']
results_df.index = ['MNIST (Dense)', 'Breast Cancer (Scattered)']
print(results_df.to_string())
print("="*50)

## Visualisation

In [ ]:
for dataset in datasets_to_test:
    dataset_name = "MNIST (Dense Image)" if dataset == 'mnist' else "Breast Cancer (Tabular)"

    plt.figure(figsize=(15, 6))

    # --- Subplot 1: Test Accuracy ---
    plt.subplot(1, 2, 1)
    for noise in noise_types_to_test:
        plt.plot(range(1, federated_rounds + 1), history[dataset][noise]['test_acc'],
                 marker='o', linewidth=2, label=f"{noise.upper()}")
    plt.title(f'{dataset_name}: Trimmed Mean Test Accuracy')
    plt.xlabel('Federated Round')
    plt.ylabel('Accuracy (%)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()

    # --- Subplot 2: Train vs Test Loss (Overfitting Check) ---
    plt.subplot(1, 2, 2)
    colors = {'none': 'blue', 'normal': 'orange', 'laplace': 'green'}

    for noise in noise_types_to_test:
        c = colors[noise]
        # Plot Test Loss (Solid Line)
        plt.plot(range(1, federated_rounds + 1), history[dataset][noise]['test_loss'],
                 marker='x', linestyle='-', linewidth=2, color=c, label=f"{noise.upper()} (Test Loss)")
        # Plot Train Loss (Dashed Line)
        plt.plot(range(1, federated_rounds + 1), history[dataset][noise]['train_loss'],
                 marker='.', linestyle='--', linewidth=1.5, color=c, label=f"{noise.upper()} (Train Loss)")

    plt.title(f'{dataset_name}: Overfitting Check (Train vs Test Loss)')
    plt.xlabel('Federated Round')
    plt.ylabel('Loss')
    plt.grid(True, linestyle='--', alpha=0.7)

    # Move legend outside the plot so it doesn't cover the lines
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout()
    plt.show()


## Empirical Conclusion: Data Modality Dictates Noise Resilience

The results of the Trimmed Mean experiment reveal a fundamental divergence in how different data structures process statistical noise.

**1. The Fragility of Dense Pixel Data (MNIST)**
While Trimmed Mean successfully facilitated a baseline accuracy of **92%** without noise, the injection of privacy noise severely disrupted the spatial relationships inherent in image data. Normal (Gaussian) noise caused a moderate drop to **76%**, but the heavy-tailed nature of Laplace noise proved catastrophic, shattering the model's accuracy down to **46%**. Dense data is highly intolerant to extreme perturbations.

**2. The Resilience of Scattered Features (Tabular)**
In stark contrast, the Tabular Breast Cancer dataset demonstrated incredible structural robustness. It achieved an exceptional **96%** baseline accuracy, and remarkably, sustained a **94%** accuracy even under extreme Laplace noise. Because tabular data features are largely independent, heavy-tailed perturbations act as a mild regularizer rather than a destructive force.

**Takeaway:** Trimmed Mean is a safe and highly effective defense, but the choice of statistical noise must be strictly paired to the data modality. Heavy-tailed Laplace noise should be exclusively reserved for tabular datasets.